In [23]:
# Install required library
!pip install bert-score -q

In [24]:
import json
import os
from bert_score import score

In [29]:
# ── File paths ────────────────────────────────────────────────────────────────
# Upload your JSONs as a Kaggle dataset and set the folder name below.
# Run  !ls /kaggle/input/  to find the exact folder name.
DATASET_FOLDER    = "datasets/chandrimanandi/summary"   # <-- change this
GROUND_TRUTH_PATH = f"/kaggle/input/{DATASET_FOLDER}/summary.json"
DRAFT_PATH        = f"/kaggle/input/{DATASET_FOLDER}/summary2.json"
OUTPUT_DIR        = "/kaggle/working/"


# ── Threshold ─────────────────────────────────────────────────────────────────
# Ops with combined_score < THRESHOLD are flagged as FAIL.
SIMILARITY_THRESHOLD = 0.80

# ── Models ────────────────────────────────────────────────────────────────────
# STS model: fine-tuned on sentence similarity, scale 0-1
STS_MODEL = "cross-encoder/stsb-roberta-large"

# NLI model: detects contradiction / entailment between sentence pairs
# This is what catches negation flips ("must NOT exist" vs "must exist")
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

# ── Score weighting ───────────────────────────────────────────────────────────
# STS captures paraphrase similarity; NLI catches logical/negation changes.
# If your ops are short and precise, increase NLI_WEIGHT.
STS_WEIGHT = 0.50
NLI_WEIGHT = 0.50

In [30]:
!pip install sentence-transformers -q

In [31]:
import json, os
from pathlib import Path
import pandas as pd
from sentence_transformers import CrossEncoder

In [32]:
def load_json(path):
    assert os.path.exists(path), f"File not found: {path}"
    with open(path) as f:
        return json.load(f)

def extract_operations(data):
    """Returns {operationId: combined_text} for every op in every section."""
    ops = {}
    for section in data["sections"]:
        for op in section["operations"]:
            op_id = op["operationId"]
            text  = " ".join(filter(None, [
                op.get("name", ""),
                op.get("purpose", ""),
                op.get("precondition", ""),
                op.get("postcondition", ""),
            ]))
            ops[op_id] = text.strip()
    return ops

ground_truth_data = load_json(GROUND_TRUTH_PATH)
draft_data        = load_json(DRAFT_PATH)

gt_ops    = extract_operations(ground_truth_data)
draft_ops = extract_operations(draft_data)

common_ids    = sorted(set(gt_ops.keys()) & set(draft_ops.keys()))
only_in_gt    = set(gt_ops.keys()) - set(draft_ops.keys())
only_in_draft = set(draft_ops.keys()) - set(gt_ops.keys())

print(f"Ground truth ops : {len(gt_ops)}")
print(f"Draft ops        : {len(draft_ops)}")
print(f"Common ops       : {len(common_ids)}")
if only_in_gt:    print(f"Missing from draft   : {only_in_gt}")
if only_in_draft: print(f"Extra in draft       : {only_in_draft}")

Ground truth ops : 21
Draft ops        : 21
Common ops       : 21


In [33]:
print(f"Loading STS model : {STS_MODEL}")
sts_model = CrossEncoder(STS_MODEL)

print(f"Loading NLI model : {NLI_MODEL}")
nli_model = CrossEncoder(NLI_MODEL, num_labels=3)

# Label order for nli-deberta-v3-base
NLI_LABELS = ["contradiction", "entailment", "neutral"]
print("Models ready.")

Loading STS model : cross-encoder/stsb-roberta-large


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Models ready.


In [34]:
pairs = [(gt_ops[i], draft_ops[i]) for i in common_ids]

# ── STS scores (0–1, higher = more similar) ───────────────────────────────────
print("Computing STS scores...")
raw_sts = sts_model.predict(pairs)          # returns raw logits for stsb
# stsb-roberta-large outputs a regression score roughly in [0, 1]
# Clip to [0,1] to be safe
sts_scores = [max(0.0, min(1.0, float(s))) for s in raw_sts]

# ── NLI scores ────────────────────────────────────────────────────────────────
print("Computing NLI scores...")
nli_logits = nli_model.predict(pairs, apply_softmax=True)  # shape (N, 3)

nli_scores      = []
nli_labels      = []
contradiction_p = []
entailment_p    = []
neutral_p       = []

for logit in nli_logits:
    c, e, n = float(logit[0]), float(logit[1]), float(logit[2])
    # NLI similarity score: full credit for entailment, half for neutral, 0 for contradiction
    nli_sim = e + 0.5 * n
    nli_scores.append(round(nli_sim, 4))
    nli_labels.append(NLI_LABELS[int(logit.argmax())])
    contradiction_p.append(round(c, 4))
    entailment_p.append(round(e, 4))
    neutral_p.append(round(n, 4))

# ── Combined score ────────────────────────────────────────────────────────────
combined_scores = [
    round(STS_WEIGHT * s + NLI_WEIGHT * n, 4)
    for s, n in zip(sts_scores, nli_scores)
]

print("Scoring complete.")

Computing STS scores...
Computing NLI scores...
Scoring complete.


In [35]:
results = pd.DataFrame({
    "operationId":      common_ids,
    "combined_score":   combined_scores,
    "status":           ["PASS" if s >= SIMILARITY_THRESHOLD else "FAIL" for s in combined_scores],
    "sts_score":        [round(s, 4) for s in sts_scores],
    "nli_score":        nli_scores,
    "nli_label":        nli_labels,
    "entailment_p":     entailment_p,
    "neutral_p":        neutral_p,
    "contradiction_p":  contradiction_p,
    "ground_truth_text": [gt_ops[i] for i in common_ids],
    "draft_text":        [draft_ops[i] for i in common_ids],
})

# ── Print summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  JUDGE RESULTS  |  threshold = {SIMILARITY_THRESHOLD}")
print(f"{'='*70}")
display_cols = ["operationId", "status", "combined_score", "sts_score", "nli_score", "nli_label"]
print(results[display_cols].to_string(index=False))

passed = results[results.status == "PASS"]
failed = results[results.status == "FAIL"]
print(f"\n  PASSED: {len(passed)}/{len(results)}")
print(f"  FAILED: {len(failed)}/{len(results)}")

if not failed.empty:
    print(f"\n  ⚠ Failed op IDs: {failed.operationId.tolist()}")

# ── Flag contradiction cases specifically ─────────────────────────────────────
contradictions = results[results.nli_label == "contradiction"]
if not contradictions.empty:
    print(f"\n  🔴 CONTRADICTION detected in {len(contradictions)} op(s):")
    for _, row in contradictions.iterrows():
        print(f"     {row.operationId}")
        print(f"       GT   : {row.ground_truth_text[:120]}")
        print(f"       Draft: {row.draft_text[:120]}")
        print(f"       contradiction_p={row.contradiction_p:.3f}  sts_score={row.sts_score:.3f}")

print(f"\n  Average combined score: {results.combined_score.mean():.4f}")


  JUDGE RESULTS  |  threshold = 0.8
operationId status  combined_score  sts_score  nli_score     nli_label
        1-A   FAIL          0.3797     0.7584     0.0011 contradiction
        1-B   PASS          0.9808     0.9688     0.9929    entailment
        1-C   PASS          0.9835     0.9723     0.9947    entailment
        2-k   PASS          0.9802     0.9672     0.9931    entailment
        3-A   PASS          0.9810     0.9724     0.9896    entailment
       3-B1   PASS          0.9797     0.9718     0.9875    entailment
       3-B2   PASS          0.9762     0.9735     0.9790    entailment
       3-B3   PASS          0.9764     0.9729     0.9799    entailment
       3-C1   PASS          0.9758     0.9726     0.9789    entailment
       3-C2   PASS          0.9782     0.9717     0.9846    entailment
       3-C3   PASS          0.9839     0.9712     0.9966    entailment
        4-A   PASS          0.9830     0.9712     0.9948    entailment
        4-B   PASS          0.9799     0

In [36]:
# Full results CSV
csv_path = os.path.join(OUTPUT_DIR, "judge_results.csv")
results.to_csv(csv_path, index=False)
print(f"Full results saved → {csv_path}")

# Failed-ops JSON — feed this back into your LLM pipeline for retry
if not failed.empty:
    failed_ids = failed.operationId.tolist()
    failed_payload = {
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "failed_operation_ids": failed_ids,
        "operations": [
            op
            for section in ground_truth_data["sections"]
            for op in section["operations"]
            if op["operationId"] in failed_ids
        ]
    }
    failed_path = os.path.join(OUTPUT_DIR, "failed_ops.json")
    with open(failed_path, "w") as f:
        json.dump(failed_payload, f, indent=2)
    print(f"Failed ops payload saved → {failed_path}")
    print("  Feed this JSON into your LLM 1→2 pipeline to regenerate only these operations.")
else:
    print("✅ All operations passed. No retry needed.")

Full results saved → /kaggle/working/judge_results.csv
Failed ops payload saved → /kaggle/working/failed_ops.json
  Feed this JSON into your LLM 1→2 pipeline to regenerate only these operations.
